# MISSION DATAPULSE SERIES | DATATHON #01
## AI-Generated vs Human-Written Text Detection

**Goal:** predict `0 = human-written` and `1 = AI-generated`, optimizing F1-score.

This notebook:
1. inspects the supplied data and label balance;
2. validates with 5-fold Stratified Cross Validation;
3. compares a character + word TF-IDF representation;
4. tunes a decision threshold using out-of-fold decision scores;
5. trains on all labeled rows and creates the required `id,label` CSV.

> The cross-validation F1 is an estimate, not a guaranteed leaderboard score. The threshold is selected from out-of-fold predictions and may still be somewhat optimistic on a small dataset.


## Why this approach?

- **Character TF-IDF (2–5 grams):** captures spelling, punctuation, word fragments and stylistic character patterns.
- **Word TF-IDF (1–3 grams):** captures individual words and short word sequences.
- **LinearSVC:** a strong, efficient classifier for sparse high-dimensional text features.
- **Feature weighting:** character features have weight 1.0 and word features 0.5.
- **Threshold tuning:** `LinearSVC` produces decision scores rather than probabilities. We select the threshold that maximizes F1 on out-of-fold scores.

We deliberately do not use the training-only engineered columns (`word_count`, `char_count`, etc.) because they are absent from the test file. The same text-derived features are therefore available for both train and test.


In [5]:
import os
import numpy as np
import pandas as pd
from scipy.sparse import hstack
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import f1_score, classification_report

# If using Colab, upload train.csv and test.csv, or change these paths.
TRAIN_PATH = '../dataset/train.csv' if os.path.exists('train.csv') else 'train(1).csv'
TEST_PATH = '../dataset/train.csv' if os.path.exists('test.csv') else 'test(1).csv'

train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)

print('Train shape:', train.shape)
print('Test shape:', test.shape)
print('Columns:', train.columns.tolist())
print('Label counts:')
print(train['label'].value_counts())

label_map = {'human': 0, 'ai': 1}
y = train['label'].astype(str).str.lower().map(label_map)
if y.isna().any():
    raise ValueError('Unexpected label value found. Expected human or ai.')
y = y.astype(int).to_numpy()

train_text = train['text'].fillna('').astype(str)
test_text = test['text'].fillna('').astype(str)


FileNotFoundError: [Errno 2] No such file or directory: 'train(1).csv'

In [ ]:
# 5-fold out-of-fold validation
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

char_vectorizer = TfidfVectorizer(
    analyzer='char',
    ngram_range=(2, 5),
    min_df=1,
    sublinear_tf=True,
    max_features=300000,
    dtype=np.float32
)
word_vectorizer = TfidfVectorizer(
    analyzer='word',
    ngram_range=(1, 3),
    min_df=2,
    sublinear_tf=True,
    max_features=150000,
    dtype=np.float32
)

X_char = char_vectorizer.fit_transform(train_text)
X_word = word_vectorizer.fit_transform(train_text)
X = hstack([X_char, X_word * 0.5], format='csr')

base_model = LinearSVC(C=0.3)
oof_scores = cross_val_predict(
    base_model, X, y, cv=cv,
    method='decision_function', n_jobs=-1
)

# Tune the decision threshold on out-of-fold scores
thresholds = np.linspace(-0.5, 0.5, 201)
threshold_f1 = [
    f1_score(y, (oof_scores >= threshold).astype(int))
    for threshold in thresholds
]
best_index = int(np.argmax(threshold_f1))
BEST_THRESHOLD = float(thresholds[best_index])

print('F1 at default threshold 0.00:', f1_score(y, (oof_scores >= 0).astype(int)))
print('Best OOF threshold:', BEST_THRESHOLD)
print('Best OOF F1:', threshold_f1[best_index])
print(classification_report(
    y, (oof_scores >= BEST_THRESHOLD).astype(int),
    target_names=['Human (0)', 'AI (1)']
))


F1 at default threshold 0.00: 0.9709864603481625
Best OOF threshold: 0.050000000000000044
Best OOF F1: 0.9737098344693281
              precision    recall  f1-score   support

   Human (0)       0.97      0.98      0.97       521
      AI (1)       0.98      0.97      0.97       518

    accuracy                           0.97      1039
   macro avg       0.97      0.97      0.97      1039
weighted avg       0.97      0.97      0.97      1039



In [ ]:
# Refit vectorizers and classifier on ALL labeled training data
final_char_vectorizer = TfidfVectorizer(
    analyzer='char', ngram_range=(2, 5), min_df=1,
    sublinear_tf=True, max_features=300000, dtype=np.float32
)
final_word_vectorizer = TfidfVectorizer(
    analyzer='word', ngram_range=(1, 3), min_df=2,
    sublinear_tf=True, max_features=150000, dtype=np.float32
)

Xc_train = final_char_vectorizer.fit_transform(train_text)
Xw_train = final_word_vectorizer.fit_transform(train_text)
X_train = hstack([Xc_train, Xw_train * 0.5], format='csr')

final_model = LinearSVC(C=0.3)
final_model.fit(X_train, y)

# Transform test data with the SAME fitted vectorizers
Xc_test = final_char_vectorizer.transform(test_text)
Xw_test = final_word_vectorizer.transform(test_text)
X_test = hstack([Xc_test, Xw_test * 0.5], format='csr')

test_scores = final_model.decision_function(X_test)
test_predictions = (test_scores >= BEST_THRESHOLD).astype(int)

submission = pd.DataFrame({
    'id': test['id'],
    'label': test_predictions
})

# Required submission validation
assert submission.columns.tolist() == ['id', 'label']
assert len(submission) == len(test)
assert submission['id'].equals(test['id'])
assert submission['id'].is_unique
assert set(submission['label'].unique()).issubset({0, 1})

submission.to_csv('datapulse_submission.csv', index=False)
print(submission.head())
print('Predicted label counts:')
print(submission['label'].value_counts().sort_index())
print('Saved: datapulse_submission.csv')


   id  label
0   1      0
1   2      1
2   3      0
3   4      1
4   5      0
Predicted label counts:
label
0    133
1    127
Name: count, dtype: int64
Saved: datapulse_submission.csv


## Submission checklist
- `datapulse_submission.csv` has exactly two columns: `id,label`.
- Every test ID is preserved exactly once and in original order.
- Labels are integers `0` or `1` only.
- Keep this notebook (`.ipynb`) with the CSV when submitting.
